# Kernel PCA finds a rejection rule

A machine shop measures two deviations on every part it makes, each scaled to lie between
-1 and 1. An inspector has been rejecting parts by a rule nobody wrote down, and the
records of 300 inspected parts are all there is. The rule turns out to be that a part is
rejected when its two deviations point the same way. No straight line in the plane of the
two deviations separates the rejected parts from the others, so linear PCA cannot help.
You will write kernel PCA from the kernel matrix up, fit it on the 300 records, and use
its first component to sort 300 new parts.

**How to work through it.** Each part asks for one or two functions. Replace the `TODO`
and the `raise NotImplementedError` with your code, then run the check cell under it. The
check prints `Part N passed` when your code is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np

rng = np.random.default_rng(52)


def inspected(n):
    D = rng.uniform(-1, 1, size=(n, 2))                 # the two scaled deviations
    return D, (D[:, 0] * D[:, 1] > 0).astype(int)        # 1 = rejected


D_train, y_train = inspected(300)
D_test, y_test = inspected(300)


def kernel(A, B):
    # the polynomial kernel (a . b)^2, the inner product of the features (a1^2, a2^2, sqrt(2) a1 a2)
    return (A @ B.T) ** 2


print(f"{len(D_train)} records, {y_train.mean():.0%} rejected; {len(D_test)} new parts")

## Part 1 (coding task): Centering in feature space

Write two functions.

- `center_train(K)` takes the $n \times n$ kernel matrix of the training points and returns
  $\tilde K$, the matrix of inner products of the features after the training mean of the
  features is subtracted.
- `center_new(K_new, K)` takes the $m \times n$ kernel values between $m$ new points and the
  training points, and returns the inner products of the new points' centered features with
  the training points' centered features. Center with the training mean, as a fitted
  transform must.

The check uses a linear kernel, whose features are the points themselves, so the right
answer can be written down directly.

In [ ]:
def center_train(K):
    # TODO
    raise NotImplementedError


def center_new(K_new, K):
    # TODO
    raise NotImplementedError

In [ ]:
r = np.random.default_rng(8)
Z = r.normal(size=(40, 3)) * [3.0, 1.0, 0.5] + 4.0     # far from the origin, so centering matters
Z_new = r.normal(size=(7, 3)) + 4.0
mu = Z.mean(axis=0)
Kc = np.asarray(center_train(Z @ Z.T), dtype=float)
assert Kc.shape == (40, 40), "center_train should return an n x n matrix"
assert np.allclose(Kc, (Z - mu) @ (Z - mu).T), "center_train does not give the inner products of the centered features"
Knc = np.asarray(center_new(Z_new @ Z.T, Z @ Z.T), dtype=float)
assert Knc.shape == (7, 40), "center_new should return one row per new point"
assert np.allclose(Knc, (Z_new - mu) @ (Z - mu).T), "center_new does not match the new points' features centered with the training mean"
one = np.asarray(center_new(Z_new[:1] @ Z.T, Z @ Z.T), dtype=float)
assert np.allclose(one, Knc[:1]), "a new point's centered row must not depend on which other new points arrive with it"
print("Part 1 passed")

## Part 2 (coding task): Fitting the components

Write `kpca_fit(K, n_components)`. Center `K`, find its eigenvalues and unit eigenvectors,
and return the `n_components` largest eigenvalues, largest first, as an array `lambdas`,
with the matching eigenvectors as the columns of an $n \times$ `n_components` array
`alphas`. The training scores on the components are then `alphas * np.sqrt(lambdas)`.

The check compares the result with ordinary PCA, which a linear kernel has to reproduce.

In [ ]:
def kpca_fit(K, n_components):
    # TODO
    raise NotImplementedError

In [ ]:
lambdas, alphas = kpca_fit(Z @ Z.T, 2)
lambdas, alphas = np.asarray(lambdas, dtype=float), np.asarray(alphas, dtype=float)
U, S, Vt = np.linalg.svd(Z - mu, full_matrices=False)
assert lambdas.shape == (2,) and alphas.shape == (40, 2), "return 2 eigenvalues and a 40 x 2 array of eigenvectors"
assert np.allclose(lambdas, S[:2] ** 2), f"eigenvalues {np.round(lambdas, 3)}: expected the two largest, {np.round(S[:2] ** 2, 3)}, largest first"
assert np.allclose(np.linalg.norm(alphas, axis=0), 1), "the eigenvectors should have unit length"
scores = alphas * np.sqrt(lambdas)
pca_scores = (Z - mu) @ Vt[:2].T
assert np.allclose(np.abs(scores), np.abs(pca_scores)), "the training scores do not match PCA, up to the sign of each component"
print("Part 2 passed")

## Part 3 (coding task): New parts on the first component

Write `kpca_transform(K_new, K, lambdas, alphas)`, which returns the scores of new points
on the fitted components, an $m \times$ `n_components` array. A new point's score on
component $c$ is its centered kernel row times `alphas[:, c]`, divided by
`np.sqrt(lambdas[c])`.

The check then fits kernel PCA with the polynomial kernel on the 300 records and scores the
new parts. A part counts as rejected when its first-component score lies on the same side
of the score of the point $(0, 0)$ as most rejected records do.

In [ ]:
def kpca_transform(K_new, K, lambdas, alphas):
    # TODO
    raise NotImplementedError

In [ ]:
K_lin = Z @ Z.T
lam_all, vec_all = np.linalg.eigh((Z - mu) @ (Z - mu).T)
lam_l, alp_l = lam_all[::-1][:2], vec_all[:, ::-1][:, :2]      # the top two components, computed here
assert np.allclose(kpca_transform(K_lin, K_lin, lam_l, alp_l), alp_l * np.sqrt(lam_l)), \
    "transforming the training points must give back their training scores"
new_scores = np.asarray(kpca_transform(Z_new @ Z.T, K_lin, lam_l, alp_l), dtype=float)
assert np.allclose(np.abs(new_scores), np.abs((Z_new - mu) @ Vt[:2].T)), "new points' scores do not match PCA's projections"

K = kernel(D_train, D_train)
lam, alp = kpca_fit(K, 3)
z_train = alp[:, 0] * np.sqrt(lam[0])
z_test = np.asarray(kpca_transform(kernel(D_test, D_train), K, lam, alp), dtype=float)[:, 0]
z_origin = float(np.asarray(kpca_transform(kernel(np.zeros((1, 2)), D_train), K, lam, alp))[0, 0])
side = 1 if ((z_train > z_origin) == y_train).mean() > 0.5 else -1
accuracy = ((side * (z_test - z_origin) > 0) == y_test).mean()
print(f"eigenvalues / n: {np.round(lam / len(K), 3)}")
print(f"accuracy on the new parts from one kernel PCA component: {accuracy:.3f}")
_mu = D_train.mean(axis=0)
_V = np.linalg.svd(D_train - _mu, full_matrices=False)[2]
for _c in range(2):
    _s, _s_new = (D_train - _mu) @ _V[_c], (D_test - _mu) @ _V[_c]
    _, _t, _g = max((((g * (_s - t) > 0) == y_train).mean(), t, g) for t in _s for g in (1, -1))
    print(f"linear PCA component {_c + 1}, threshold set on the records: "
          f"accuracy {((_g * (_s_new - _t) > 0) == y_test).mean():.3f} on the new parts")
assert accuracy > 0.97, f"accuracy {accuracy:.3f}: the first component should sort the new parts almost perfectly"
print("Part 3 passed")

## Solution

### Part 1: Centering in feature space

In [ ]:
def center_train(K):
    return K - K.mean(axis=0, keepdims=True) - K.mean(axis=1, keepdims=True) + K.mean()


def center_new(K_new, K):
    col = K.mean(axis=0, keepdims=True)          # training column means, one per training point
    own = K_new.mean(axis=1, keepdims=True)      # each new row's own mean over the training points
    return K_new - col - own + K.mean()

**Why it works.** With $\mu$ the mean training feature,
$(\phi_i - \mu)^\top(\phi_j - \mu) = K_{ij} - \frac1n \sum_k K_{kj} - \frac1n \sum_k K_{ik} + \frac1{n^2} \sum_{k,l} K_{kl}$:
the entry minus its column mean and its row mean, plus the grand mean. For a new point the
same expansion holds with $\phi_i$ replaced by the new feature, so its row loses its own
mean over the training points and the training column means, and gains the training grand
mean. Centering a batch of new rows with the batch's own column means would make each
point's result depend on its batch mates. A batch of one would come out as all zeros.

### Part 2: Fitting the components

In [ ]:
def kpca_fit(K, n_components):
    lam, vecs = np.linalg.eigh(center_train(K))           # eigh returns ascending eigenvalues
    order = np.argsort(lam)[::-1][:n_components]
    return lam[order], vecs[:, order]

**Why it works.** `np.linalg.eigh` returns the eigenvalues of a symmetric matrix in ascending order, so the
first columns it gives are the smallest components, here zeros. Sorting puts the largest
first. Each eigenvalue of $\tilde K$ is $n$ times the variance of the training points
along that component in feature space, and with a linear kernel it is the square of a
singular value of the centered data, which is why the check can compare with the SVD.

### Part 3: New parts on the first component

In [ ]:
def kpca_transform(K_new, K, lambdas, alphas):
    return center_new(K_new, K) @ alphas / np.sqrt(lambdas)

**Why it works.** A training point's centered row is a row of $\tilde K$, and
$\tilde K \mathbf a / \sqrt\lambda = \lambda \mathbf a / \sqrt\lambda = \sqrt\lambda\,\mathbf a$,
so the formula gives the training points back their own scores. The kernel $(a \cdot b)^2$
has the features $a_1^2$, $a_2^2$ and $\sqrt2\, a_1 a_2$. For deviations spread evenly over
the square, $\sqrt2\, a_1 a_2$ has variance $2/9 = 0.222$ against $4/45 = 0.089$ for each
square, so the first component is the product $a_1 a_2$, up to sign, and the eigenvalues
over $n$ come out near those values. The point $(0, 0)$ has the feature vector 0, so the
side of its score is the sign of the product, which is the inspector's rule. One component
sorts the new parts with an accuracy of 0.997. A threshold on either linear PCA component,
set on the records, gets 0.480 and 0.467, no better than chance.